## Gemini File Search

In [ ]:
import os
import time
from google import genai
from google.genai import types

In [ ]:
# Make sure the GOOGLE_API_KEY environment variable is set
client = genai.Client()

### Create a File Search Store

A store is the persistent container holding the embeddings for your documents.

In [ ]:
# Create the File Search store with an optional display name
file_search_store = client.file_search_stores.create(
    config={'display_name': 'my-rag-store'}
)

print("Created store:", file_search_store.name)

In [ ]:
# Replace with your local file path. Supported: PDF, DOCX, TXT, JSON, many code files.
local_path = "../course-selector-opt2.xlsx"

### Upload and index a file

You have two common paths:

- A) Direct upload that also indexes (simplest)

In [ ]:
operation = client.file_search_stores.upload_to_file_search_store(
    file=local_path,
    file_search_store_name='fileSearchStores/myragstore-vm45fup8l4oj',
    config={
        'display_name': os.path.basename(local_path),   # appears in citations
        # Optional advanced chunking:
        # 'chunking_config': {
        #   'white_space_config': {
        #     'max_tokens_per_chunk': 300,
        #     'max_overlap_tokens': 30
        #   }
        # },
        # Optional metadata for filtering later:
        # 'custom_metadata': [
        #   {'key': 'doc_type', 'string_value': 'manual'},
        #   {'key': 'version', 'string_value': 'v2'}
        # ]
    }
)

# Poll until indexing completes
while not operation.done:
    time.sleep(2)
    operation = client.operations.get(operation)

print("Upload+index complete.")

- B) Upload via Files API, then import to store (useful if you already use Files API)

In [ ]:
sample_file = client.files.upload(file=local_path, config={'name': os.path.basename(local_path)})

operation = client.file_search_stores.import_file(
    file_search_store_name='fileSearchStores/myragstore-vm45fup8l4oj',
    file_name=os.path.basename(local_path),
    # custom_metadata=[...],
    # chunking_config={...}
)

while not operation.done:
    time.sleep(2)
    operation = client.operations.get(operation)

print("Import+index complete from Files API.")

Notes: Temporary Files API uploads are auto-deleted after 48 hours; the embedded data in your File Search Store persists until you delete it. See File Search docs.

## Ask questions grounded on your store

Use generate_content and pass the fileSearch tool with your store name.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Summarize the safety precautions from our manual.",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=['fileSearchStores/myragstore-vm45fup8l4oj']
                )
            )
        ]
    )
)

print("Answer:\n", response.text)

# Optional: view citations/grounding metadata
gm = response.candidates[0].grounding_metadata if response.candidates else None
print("\nGrounding metadata (citations):\n", gm)

## Restrict retrieval using metadata filters (optional)

If you attached metadata during import, you can filter to a subset.

In [ ]:
filtered = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What steps are required to reset the device?",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=['fileSearchStores/myragstore-vm45fup8l4oj'],
                    metadata_filter='doc_type="manual"'   # AIP-160 syntax
                )
            )
        ]
    )
)
print("\nFiltered answer:\n", filtered.text)

## Find, list, and delete documents in the store (optional)

In [ ]:
# List documents
pager = client.file_search_stores.documents.list(parent='fileSearchStores/myragstore-vm45fup8l4oj')
for doc in pager.page:
    print("Document:", doc.display_name, "| name:", doc.name)

# Delete a specific document (required to "update" it)
# client.file_search_stores.documents.delete(
#     name=doc.name,
#     config={'force': True}
# )

## Clean up the store (optional)

In [ ]:
# Delete the whole store (and its docs)
client.file_search_stores.delete(
    name='fileSearchStores/myragstore-vm45fup8l4oj',
    config={'force': True}  # force deletes indexed docs, too
)

print("Deleted store:", 'fileSearchStores/myragstore-vm45fup8l4oj')